# Predict the first compound in a dataset
Run `01_preprocess.ipynb` first. Select a dataset to predict the RT of its first compound using 10 similar compounds from the remaining references. Compare the prediction with the measured RT.

In [1]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "rt_icl").is_dir() or not (ROOT / "data/raw").is_dir():
    raise RuntimeError("Open this notebook with the release folder as the working directory.")
RAW_DIR = ROOT / "data/raw"
DATA_DIR = ROOT / "data/processed"
RESULTS_DIR = ROOT / "results"

## Model and run settings
Supported choices:

| Model |
|---|
| `gemini-3-flash-preview` |
| `gpt-5.4-mini-2026-03-17` |
| `qwen/qwen3-235b-a22b-2507` |
| `openai/gpt-oss-120b` |

In [2]:
from rt_icl import (
    load_dataset, prepare_inference_requests, create_provider, run_inference,
)

DATASET_ID = "0004"
PROVIDER = "gemini"
MODEL = "gemini-3-flash-preview"

## Load the dataset
The first compound is the prediction target; the remaining compounds are references. LC conditions are loaded from the same dataset.

In [3]:
data = load_dataset(DATASET_ID, DATA_DIR)
query = data.compounds[0]
references = data.compounds[1:]
queries = [query]
lc_condition = data.lc_condition

display({"Name": query.get("Name", ""), "SMILES": query["SMILES"], "measured_rt_seconds": query["RT"]})

{'Name': '3-hydroxy-3-methylbutyric acid (3-hydroxyisovaleric acid) ',
 'SMILES': 'CC(C)(O)CC(=O)O',
 'measured_rt_seconds': 252.0}

## Preview a prompt

In [4]:
requests = prepare_inference_requests(references, queries, lc_condition)
print(requests[0].prompt.user_content)

You are an expert analytical chemist specializing in Liquid Chromatography–Mass Spectrometry (LC–MS).
Your task is to predict the retention time (RT) of a query compound under the specified target chromatographic system.
You are provided with the following information:
- The LC conditions of the target chromatographic system.
- A query compound whose RT is unknown.
- A set of structurally similar reference compounds along with their measured RTs.
- Relevant molecular identifiers and descriptors for the query and reference compounds. 

<Conditions>
LC mode: Reversed-Phase
Column: Thermo Scientific Hypersil GOLD (2.1 mm × 150 mm, 1.9 μm)
Mobile phase A: 0.1% formic acid in water (pH 3)
Mobile phase B: 0.1% formic acid in acetonitrile (pH 3)
Flow rate: 0.5 mL/min
Gradient:
  0 min (0 s): 100% A, 0% B
  2 min (120 s): 100% A, 0% B
  13 min (780 s): 0% A, 100% B
  15.5 min (930 s): 0% A, 100% B
  19 min (1140 s): 100% A, 0% B
</Conditions>

- Use the reference compounds and their measured R

## API keys
Copy `.env.example` to `.env` in the project folder if you do not already have one. Enter the API key for your selected `PROVIDER`: `GOOGLE_API_KEY`, `OPENAI_API_KEY`, or `OPENROUTER_API_KEY`. Then run this cell to load it.

In [5]:
import os
from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)

key_name = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "openrouter": "OPENROUTER_API_KEY"}[PROVIDER]
if not os.environ.get(key_name):
    raise ValueError(f"Set {key_name} in the project .env file.")

## Execute prediction
The next cell makes one prediction request. Existing results are never overwritten; use a new output location to repeat it.

In [6]:
import re

provider = create_provider(PROVIDER, model=MODEL)
model_tag = re.sub(r"[^A-Za-z0-9._-]+", "_", provider.model).strip("._-") or "model"
output_dir = RESULTS_DIR / "compound" / DATASET_ID / PROVIDER / model_tag
result = await run_inference(
    references, queries, lc_condition, provider,
    max_concurrent=1, output_dir=output_dir,
)

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


## Compare measured and predicted RT

In [7]:
prediction = result.predictions[0]
predicted_rt = prediction["pred_rt"]
display({
    "Name": query.get("Name", ""),
    "SMILES": query["SMILES"],
    "measured_rt_seconds": query["RT"],
    "predicted_rt_seconds": predicted_rt,
    "absolute_error_seconds": abs(predicted_rt - query["RT"]) if predicted_rt is not None else None,
})
if prediction["error"]:
    print("Prediction failed:", prediction["error"])
print("Saved to:", result.output_dir)

{'Name': '3-hydroxy-3-methylbutyric acid (3-hydroxyisovaleric acid) ',
 'SMILES': 'CC(C)(O)CC(=O)O',
 'measured_rt_seconds': 252.0,
 'predicted_rt_seconds': 264.0,
 'absolute_error_seconds': 12.0}

Saved to: /home/dm3/woojae/00_RT/RT-ICL/results/compound/0004/gemini/gemini-3-flash-preview


RTs are in seconds. Each run saves `run_metadata.json`, `metrics.json`, and `predictions.csv` under `results/compound/`. The target’s measured RT is used only for evaluation and is excluded from the prompt.